<a href="https://colab.research.google.com/github/nathanpua/agent-rl/blob/main/2048-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train Qwen 3.5 9B on 2048 — Colab T4

Uses ART `LocalBackend` on a free T4 GPU. The model is loaded in **4-bit** to fit in 16GB VRAM.

### Setup
1. Runtime > Change runtime type > **T4 GPU**
2. Set your `WANDB_API_KEY` in the secrets panel (key icon in left sidebar)
3. Runtime > Run all

In [ ]:
%pip install openpipe-art==0.5.17 transformers peft accelerate bitsandbytes sentencepiece vllm -U -q

In [ ]:
import os

# Load WANDB_API_KEY from Colab secrets
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    pass  # Already set or not needed

if not os.getenv("WANDB_API_KEY"):
    print("WARNING: WANDB_API_KEY not set. Training will work but metrics won't be logged.")
    print("Set it in: Colab left sidebar > Key icon > Add WANDB_API_KEY")
else:
    print("WANDB_API_KEY found.")

### Game Environment

In [ ]:
import random
import string
import xml.etree.ElementTree as ET
from typing import Literal, TypedDict

WINNING_VALUE = 64


class TwentyFortyEightGame(TypedDict):
    id: str
    board: list[list[int | None]]


def populate_random_cell(game: TwentyFortyEightGame) -> None:
    all_clear_coordinates = [
        (i, j)
        for i in range(len(game["board"]))
        for j in range(len(game["board"][i]))
        if game["board"][i][j] is None
    ]
    random_clear_coordinates = random.choice(all_clear_coordinates)
    game["board"][random_clear_coordinates[0]][random_clear_coordinates[1]] = (
        2 if random.random() < 0.9 else 4
    )


def generate_game(board_length: int = 4) -> TwentyFortyEightGame:
    id = "".join(random.choices(string.ascii_letters + string.digits, k=6))
    game = {
        "id": id,
        "board": [[None for _ in range(board_length)] for _ in range(board_length)],
    }
    populate_random_cell(game)
    populate_random_cell(game)
    return game


def render_board(game: TwentyFortyEightGame) -> str:
    board = game["board"]
    max_cell_width = max(
        [len(str(cell)) for row in board for cell in row if cell is not None]
    )
    board_str = ""
    for row in board:
        board_str += "|".join(
            [
                str(cell).rjust(max_cell_width)
                if cell is not None
                else "_".rjust(max_cell_width)
                for cell in row
            ]
        )
        board_str += "\n"
    return board_str


def condense_sequence(sequence: list[int | None]) -> list[int | None]:
    condensed_sequence = []
    gapless_sequence = [cell for cell in sequence if cell is not None]
    i = 0
    while i < len(gapless_sequence):
        if (
            i + 1 < len(gapless_sequence)
            and gapless_sequence[i] == gapless_sequence[i + 1]
        ):
            condensed_sequence.append(gapless_sequence[i] * 2)
            i += 2
        else:
            condensed_sequence.append(gapless_sequence[i])
            i += 1
    return condensed_sequence + [None] * (4 - len(condensed_sequence))


def condense_board(
    game: TwentyFortyEightGame, direction: Literal["left", "right", "up", "down"]
) -> None:
    if direction == "left":
        for row in game["board"]:
            condensed_row = condense_sequence(row)
            for i in range(len(row)):
                row[i] = condensed_row[i]
    if direction == "right":
        for row in game["board"]:
            reversed_row = row[::-1]
            condensed_row = condense_sequence(reversed_row)[::-1]
            for i in range(len(row)):
                row[i] = condensed_row[i]
    if direction == "up":
        for col_index in range(len(game["board"][0])):
            column = [row[col_index] for row in game["board"]]
            condensed_column = condense_sequence(column)
            for row_index in range(len(column)):
                game["board"][row_index][col_index] = condensed_column[row_index]
    if direction == "down":
        for col_index in range(len(game["board"][0])):
            column = [row[col_index] for row in game["board"]]
            reversed_column = column[::-1]
            condensed_column = condense_sequence(reversed_column)[::-1]
            for row_index in range(len(column)):
                game["board"][row_index][col_index] = condensed_column[row_index]


def apply_agent_move(game: TwentyFortyEightGame, move_xml: str) -> None:
    direction = None
    try:
        root = ET.fromstring(move_xml)
        direction = root.text
    except Exception:
        raise ValueError("Invalid xml")
    if direction not in ["left", "right", "up", "down"]:
        raise ValueError("Invalid direction")
    condense_board(game, direction)
    populate_random_cell(game)


def max_cell_value(game: TwentyFortyEightGame) -> int:
    return max([cell for row in game["board"] for cell in row if cell is not None])


def check_game_finished(game: TwentyFortyEightGame) -> bool:
    if max_cell_value(game) >= WINNING_VALUE:
        return True
    if any(cell is None for row in game["board"] for cell in row):
        return False
    return True


def total_board_value(game: TwentyFortyEightGame) -> int:
    return sum([cell for row in game["board"] for cell in row if cell is not None])


print("Game environment ready.")

### Model Setup

Qwen 3.5 9B loaded in **4-bit** to fit on T4 (16GB VRAM). Both vLLM inference and LoRA training share the same GPU.

In [ ]:
import art
from art.dev import InternalModelConfig, EngineArgs
from art.local import LocalBackend

random.seed(42)

model = art.TrainableModel(
    name="qwen35-2048-colab",
    project="2048",
    base_model="Qwen/Qwen3.5-9B",
    _internal_config=InternalModelConfig(
        trainer_gpu_ids=[0],
        inference_gpu_ids=[0],
        engine_args=EngineArgs(
            gpu_memory_utilization=0.5,  # T4 has 16GB, use half for vLLM
            max_model_len=1024,          # Reduced from 2048 to save memory
            max_num_seqs=4,              # Fewer parallel sequences
            enforce_eager=True,
        ),
        init_args={
            "load_in_4bit": True,       # 4-bit to fit 9B model in 16GB
            "max_seq_length": 1024,
        },
    ),
)

backend = LocalBackend()
await model.register(backend)

print(f"Model registered: {model.name}")
print(f"Base model: {model.base_model}")
print(f"Inference name: {model.get_inference_name()}")

### Rollout

In [ ]:
import math
import requests
from openai import AsyncOpenAI
from pydantic import BaseModel


class Scenario2048(BaseModel):
    step: int


@art.retry(exceptions=(requests.ReadTimeout,))
async def rollout(model: art.Model, scenario: Scenario2048) -> art.Trajectory:
    client = AsyncOpenAI(
        base_url=model.inference_base_url,
        api_key=model.inference_api_key,
    )
    game = generate_game()
    move_number = 0

    trajectory = art.Trajectory(
        messages_and_choices=[
            {
                "role": "system",
                "content": (
                    "You are an excellent 2048 player. Always choose the move most likely to "
                    "lead to combine cells to eventually reach the number 2048. Optional moves "
                    "are 'left', 'right', 'up', 'down'. Return your move as an XML object with "
                    "a single property 'move', like so: <move>left</move>"
                ),
            }
        ],
        metadata={"game_id": game["id"], "step": scenario.step},
        reward=0,
    )

    while True:
        trajectory.messages_and_choices.append(
            {"role": "user", "content": render_board(game)}
        )

        try:
            messages = trajectory.messages()
            chat_completion = await client.chat.completions.create(
                max_completion_tokens=128,
                messages=messages,
                model=model.get_inference_name(),
            )
        except Exception as e:
            print(f"Exception during inference: {e}")
            raise e

        choice = chat_completion.choices[0]
        content = choice.message.content
        assert isinstance(content, str)
        trajectory.messages_and_choices.append(choice)

        try:
            apply_agent_move(game, content)
            move_number += 1
        except ValueError:
            trajectory.reward = -1
            break

        if check_game_finished(game):
            max_value = max_cell_value(game)
            board_value = total_board_value(game)
            trajectory.metrics["max_value"] = max_value
            trajectory.metrics["board_value"] = board_value
            trajectory.metrics["move_number"] = move_number

            if max_value < WINNING_VALUE:
                max_value_reward = (math.log(max_value, 2) - 1) / (
                    math.log(WINNING_VALUE, 2) - 1
                )
                board_value_reward = (math.log(board_value, 2) - 1) / (
                    math.log(WINNING_VALUE * 16, 2) - 1
                )
                trajectory.reward = max_value_reward + (board_value_reward * 0.2)
            else:
                trajectory.reward = 2
            break

    return trajectory


print("Rollout function ready.")

### Training Loop

Quick test: **5 steps x 6 games** (vs 20 x 18 on Vast.ai). Full training runs on Vast.ai.

In [ ]:
TRAINING_STEPS = int(os.environ.get("TRAINING_STEPS", "5"))
GAMES_PER_STEP = int(os.environ.get("GAMES_PER_STEP", "6"))

print(f"Training {TRAINING_STEPS} steps x {GAMES_PER_STEP} games/step")
print(f"This is a quick test. Full 20x18 training runs on Vast.ai.\n")

for i in range(await model.get_step(), TRAINING_STEPS):
    print(f"\n{'='*50}")
    print(f"Step {i + 1}/{TRAINING_STEPS}")
    print(f"{'='*50}")

    train_groups = await art.gather_trajectory_groups(
        (
            art.TrajectoryGroup(
                rollout(model, Scenario2048(step=i)) for _ in range(GAMES_PER_STEP)
            )
            for _ in range(1)
        ),
        pbar_desc="gather",
        max_exceptions=GAMES_PER_STEP,
    )
    await model.delete_checkpoints("train/reward")

    result = await backend.train(model, train_groups, learning_rate=5e-6)
    await model.log(
        train_groups, metrics=result.metrics, step=result.step, split="train"
    )

print(f"\n{'='*50}")
print("Training complete!")
print(f"{'='*50}")

### Play a Test Game

Watch the trained model play 2048.

In [ ]:
last_step = await model.get_step()
inference_name = f"{model.get_inference_name()}"

client = AsyncOpenAI(
    base_url=model.inference_base_url,
    api_key=model.inference_api_key,
)

game = generate_game()
move_number = 0

while not check_game_finished(game):
    messages = [
        {"role": "system", "content": "You are an excellent 2048 player. Always choose the move most likely to lead to combine cells to eventually reach the number 2048. Optional moves are 'left', 'right', 'up', 'down'. Return your move as an XML object with a single property 'move', like so: <move>left</move>"},
        {"role": "user", "content": render_board(game)},
    ]

    try:
        response = await client.chat.completions.create(
            model=inference_name,
            messages=messages,
            max_completion_tokens=256,
        )
        content = response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        break

    try:
        apply_agent_move(game, content)
        move_number += 1
    except ValueError:
        print(f"Invalid move: {content}")
        break

    if move_number % 5 == 0:
        print(f"--- move {move_number} ---")
        print(render_board(game))

max_value = max_cell_value(game)
won = max_value >= WINNING_VALUE
print(f"\n{'WON' if won else 'LOST'} in {move_number} moves")
print(f"Max tile: {max_value}")
print(f"Board value: {total_board_value(game)}")
print(f"\nFinal board:\n{render_board(game)}")